# Last Mile Operations - Analytics Notebook (LATAM)

This notebook provides step-by-step SQL queries to complete the Last Mile Logistics Assessment for LATAM operations. It uses the `google-cloud-bigquery` library with interactive user credentials authentication. All SQL blocks contained herein are standard SQL and can be copied directly into the GCP BigQuery Console GUI.

In [ ]:
# 1. Setup and Import Libraries
import os
import pandas as pd
import pydata_google_auth
from google.cloud import bigquery

In [ ]:
# 2. Interactive Google Cloud Authentication
print("Initiating browser-based OAuth token flow...")
credentials = pydata_google_auth.get_user_credentials(
    scopes=['https://www.googleapis.com/auth/cloud-platform'],
)

# Initialize BigQuery Client
project_id = "meli-last-mile-sql-assessment"
dataset_id = "LAstmile"
client = bigquery.Client(project=project_id, credentials=credentials)
print("BigQuery client initialized successfully.")

## QUESTION 1: Data Audit & Validation

This query performs a unified audit across our logistics data warehouse, calculating error rates and counts for primary key duplicates, chronological anomalies, log corruptions, missing references, and deprecated carrier contracts.

In [ ]:
q1_audit_query = """
-- 1. Duplicidad de registros en envios
SELECT 
  'shipments_new PK Duplication' as audit_check,
  COUNT(shipment_id) as total_records,
  COUNT(shipment_id) - COUNT(DISTINCT shipment_id) as error_count,
  ROUND((COUNT(shipment_id) - COUNT(DISTINCT shipment_id)) / COUNT(shipment_id) * 100, 2) as error_pct
FROM `meli-last-mile-sql-assessment.LAstmile.shipments_new`

UNION ALL

-- 2. Rutas nuevas con cronologia invertida (Fin antes de Inicio)
SELECT
  'routes_new Chrono Violations' as audit_check,
  COUNT(*) as total_records,
  COUNTIF(actual_route_end_time < actual_route_start_time) as error_count,
  ROUND(COUNTIF(actual_route_end_time < actual_route_start_time) / COUNT(*) * 100, 2) as error_pct
FROM `meli-last-mile-sql-assessment.LAstmile.routes_new`

UNION ALL

-- 3. Inconsistencias de hora en el log de eventos
SELECT
  'shipment_events_new Hour Corruption' as audit_check,
  COUNT(*) as total_records,
  COUNTIF(event_hour_utc != EXTRACT(HOUR FROM event_timestamp)) as error_count,
  ROUND(COUNTIF(event_hour_utc != EXTRACT(HOUR FROM event_timestamp)) / COUNT(*) * 100, 2) as error_pct
FROM `meli-last-mile-sql-assessment.LAstmile.shipment_events_new`
WHERE event_timestamp IS NOT NULL

UNION ALL

-- 4. Rutas nuevas sin chofer/transportista asignado
SELECT
  'routes_new Missing Partners' as audit_check,
  COUNT(*) as total_records,
  COUNTIF(partner_id IS NULL) as error_count,
  ROUND(COUNTIF(partner_id IS NULL) / COUNT(*) * 100, 2) as error_pct
FROM `meli-last-mile-sql-assessment.LAstmile.routes_new`

UNION ALL

-- 5. Contratos marcados como inactivos/invalidos (-1) en partners
SELECT
  'partners Deprecated Contracts' as audit_check,
  COUNT(*) as total_records,
  COUNTIF(active_flag = -1) as error_count,
  ROUND(COUNTIF(active_flag = -1) / COUNT(*) * 100, 2) as error_pct
FROM `meli-last-mile-sql-assessment.LAstmile.partners`

ORDER BY error_pct DESC;
"""

df_q1 = client.query(q1_audit_query).to_dataframe()
df_q1

## QUESTION 2: Productivity - Route Utilization

This section calculates stops efficiency and vehicle capacity utilization by country, applying dynamic deduplication on shipments, and profiles underperforming routes (Stops Efficiency < 60%) in Brazil.

In [ ]:
q2_productivity_query = """
WITH deduped_shipments AS (
  SELECT 
    shipment_id, 
    route_id
  FROM (
    SELECT 
      shipment_id, 
      route_id,
      ROW_NUMBER() OVER(PARTITION BY shipment_id ORDER BY status_change_timestamp DESC, delivery_attempt_count DESC) as rn
    FROM `meli-last-mile-sql-assessment.LAstmile.shipments_new`
  )
  WHERE rn = 1
),
route_shipment_counts AS (
  SELECT 
    route_id, 
    COUNT(shipment_id) as shipment_count
  FROM deduped_shipments
  GROUP BY route_id
)
SELECT 
  dc.country,
  COUNT(DISTINCT r.route_id) as total_completed_routes,
  SUM(r.estimated_stops) as total_estimated_stops,
  SUM(r.actual_stops) as total_actual_stops,
  ROUND(SAFE_DIVIDE(SUM(r.actual_stops), SUM(r.estimated_stops)) * 100, 2) as stops_efficiency_pct,
  SUM(s.shipment_count) as total_shipments_carried,
  SUM(vt.max_capacity_units) as total_max_capacity,
  ROUND(SAFE_DIVIDE(SUM(s.shipment_count), SUM(vt.max_capacity_units)) * 100, 2) as capacity_utilization_pct
FROM `meli-last-mile-sql-assessment.LAstmile.routes_new` r
JOIN `meli-last-mile-sql-assessment.LAstmile.distribution_centers` dc 
  ON r.center_id = dc.center_id
JOIN `meli-last-mile-sql-assessment.LAstmile.vehicle_types` vt 
  ON r.vehicle_type_id = vt.vehicle_type_id
LEFT JOIN route_shipment_counts s 
  ON r.route_id = s.route_id
WHERE r.route_type = 'DELIVERY'
  AND r.route_status = 'COMPLETED'
  AND r.route_date BETWEEN '2025-04-01' AND '2025-05-31'
GROUP BY dc.country
ORDER BY stops_efficiency_pct DESC;
"""

print("Computing Regional Productivity Metrics...")
df_q2 = client.query(q2_productivity_query).to_dataframe()
df_q2

In [ ]:
# Underperforming routes zoom in Brazil (top 5 worst routes by stops efficiency)
q2_brazil_outliers_query = """
WITH deduped_shipments AS (
  SELECT 
    shipment_id, 
    route_id
  FROM (
    SELECT 
      shipment_id, 
      route_id,
      ROW_NUMBER() OVER(PARTITION BY shipment_id ORDER BY status_change_timestamp DESC, delivery_attempt_count DESC) as rn
    FROM `meli-last-mile-sql-assessment.LAstmile.shipments_new`
  )
  WHERE rn = 1
),
route_shipment_counts AS (
  SELECT 
    route_id, 
    COUNT(shipment_id) as shipment_count
  FROM deduped_shipments
  GROUP BY route_id
)
SELECT 
  r.route_id,
  p.partner_name,
  vt.vehicle_type_name,
  r.estimated_stops,
  r.actual_stops,
  ROUND(SAFE_DIVIDE(r.actual_stops, r.estimated_stops) * 100, 2) as stops_efficiency_pct,
  COALESCE(s.shipment_count, 0) as shipments_carried,
  vt.max_capacity_units as vehicle_capacity
FROM `meli-last-mile-sql-assessment.LAstmile.routes_new` r
JOIN `meli-last-mile-sql-assessment.LAstmile.distribution_centers` dc 
  ON r.center_id = dc.center_id
JOIN `meli-last-mile-sql-assessment.LAstmile.vehicle_types` vt 
  ON r.vehicle_type_id = vt.vehicle_type_id
LEFT JOIN `meli-last-mile-sql-assessment.LAstmile.partners` p
  ON r.partner_id = p.partner_id
LEFT JOIN route_shipment_counts s 
  ON r.route_id = s.route_id
WHERE r.route_type = 'DELIVERY'
  AND r.route_status = 'COMPLETED'
  AND dc.country = 'BR'
  AND r.route_date BETWEEN '2025-04-01' AND '2025-05-31'
  AND SAFE_DIVIDE(r.actual_stops, r.estimated_stops) < 0.60
ORDER BY stops_efficiency_pct ASC, r.estimated_stops DESC
LIMIT 5;
"""

print("Retrieving worst 5 routes in Brazil...")
df_q2_brazil = client.query(q2_brazil_outliers_query).to_dataframe()
df_q2_brazil

## QUESTION 3: Effectiveness - Delivery Success Rate

Calculates the shipment delivery success rate (`success_rate_pct`) by partner and country, highlighting key operators and demonstrating statistical homogeneity (which points to a synthetic data footprint).

In [ ]:
q3_effectiveness_query = """
WITH deduped_shipments AS (
  SELECT 
    shipment_id, 
    route_id,
    last_status_detail
  FROM (
    SELECT 
      shipment_id, 
      route_id,
      last_status_detail,
      ROW_NUMBER() OVER(PARTITION BY shipment_id ORDER BY status_change_timestamp DESC, delivery_attempt_count DESC) as rn
    FROM `meli-last-mile-sql-assessment.LAstmile.shipments_new`
  )
  WHERE rn = 1
)
SELECT 
  dc.country,
  r.partner_id,
  p.partner_name,
  COUNT(s.shipment_id) as total_shipments,
  COUNT(CASE WHEN s.last_status_detail = 'delivered' THEN 1 END) as delivered_shipments,
  ROUND(SAFE_DIVIDE(COUNT(CASE WHEN s.last_status_detail = 'delivered' THEN 1 END), COUNT(s.shipment_id)) * 100, 2) as success_rate_pct
FROM deduped_shipments s
JOIN `meli-last-mile-sql-assessment.LAstmile.routes_new` r
  ON s.route_id = r.route_id
JOIN `meli-last-mile-sql-assessment.LAstmile.distribution_centers` dc
  ON r.center_id = dc.center_id
JOIN `meli-last-mile-sql-assessment.LAstmile.partners` p
  ON r.partner_id = p.partner_id
WHERE r.route_type = 'DELIVERY'
  AND r.route_status = 'COMPLETED'
  AND r.route_date BETWEEN '2025-04-01' AND '2025-05-31'
GROUP BY dc.country, r.partner_id, p.partner_name
ORDER BY dc.country, success_rate_pct DESC;
"""

print("Computing Success Rates by Partner & Country...")
df_q3 = client.query(q3_effectiveness_query).to_dataframe()
df_q3

## QUESTION 4: On-Time Handling (OTH)

Calculates schedule compliance (OTH) under two different operational definitions: OTH by planned end time (arrived on or before planned end hour) and OTH by duration (actual trip duration did not exceed planned duration).

In [ ]:
q4_oth_query = """
SELECT 
  dc.country,
  p.partner_name,
  vt.vehicle_type_name,
  COUNT(*) as total_routes,
  
  -- Definición 1: Finalización en la hora planificada o antes
  COUNTIF(actual_route_end_time <= planned_route_end_time) as on_time_end_count,
  ROUND(COUNTIF(actual_route_end_time <= planned_route_end_time) / COUNT(*) * 100, 2) as oth_end_pct,
  
  -- Definición 2: Duración real <= Duración planificada
  COUNTIF(
    TIMESTAMP_DIFF(actual_route_end_time, actual_route_start_time, MINUTE) <= 
    TIMESTAMP_DIFF(planned_route_end_time, planned_route_start_time, MINUTE)
  ) as on_time_duration_count,
  ROUND(COUNTIF(
    TIMESTAMP_DIFF(actual_route_end_time, actual_route_start_time, MINUTE) <= 
    TIMESTAMP_DIFF(planned_route_end_time, planned_route_start_time, MINUTE)
  ) / COUNT(*) * 100, 2) as oth_duration_pct
FROM `meli-last-mile-sql-assessment.LAstmile.routes_new` r
JOIN `meli-last-mile-sql-assessment.LAstmile.partners` p
  ON r.partner_id = p.partner_id
JOIN `meli-last-mile-sql-assessment.LAstmile.vehicle_types` vt 
  ON r.vehicle_type_id = vt.vehicle_type_id
JOIN `meli-last-mile-sql-assessment.LAstmile.distribution_centers` dc
  ON r.center_id = dc.center_id
WHERE r.route_type = 'DELIVERY'
  AND r.route_status = 'COMPLETED'
  AND r.route_date BETWEEN '2025-04-01' AND '2025-05-31'
  -- Excluimos errores de cronología
  AND r.actual_route_end_time >= r.actual_route_start_time
  AND r.planned_route_end_time >= r.planned_route_start_time
GROUP BY dc.country, p.partner_name, vt.vehicle_type_name
ORDER BY oth_end_pct ASC;
"""

print("Computing On-Time Handling (OTH) metrics...")
df_q4 = client.query(q4_oth_query).to_dataframe()
df_q4

## QUESTION 5: Timezone Shifts & Operational Patterns

Verifies whether late-night deliveries in Colombia and Brazil are real or just a UTC timezone offset logging artifact, converting UTC log dates dynamically using local DC offset factors.

In [ ]:
q5_timezone_query = """
WITH delivery_events AS (
  SELECT 
    dc.country,
    se.event_timestamp,
    dc.timezone_offset,
    -- Hora en UTC-0
    EXTRACT(HOUR FROM se.event_timestamp) as hour_utc,
    -- Hora en tiempo local (ajustada por offset)
    EXTRACT(HOUR FROM TIMESTAMP_ADD(se.event_timestamp, INTERVAL dc.timezone_offset HOUR)) as hour_local
  FROM `meli-last-mile-sql-assessment.LAstmile.shipment_events_new` se
  JOIN `meli-last-mile-sql-assessment.LAstmile.routes_new` r
    ON se.route_id = r.route_id
  JOIN `meli-last-mile-sql-assessment.LAstmile.distribution_centers` dc
    ON r.center_id = dc.center_id
  WHERE se.event_type = 'delivered'
)
SELECT 
  country,
  timezone_offset,
  COUNT(*) as total_deliveries,
  
  -- Entregas después de las 20:00 en UTC
  COUNTIF(hour_utc >= 20) as deliveries_utc_after_20,
  ROUND(COUNTIF(hour_utc >= 20) / COUNT(*) * 100, 2) as pct_utc_after_20,
  
  -- Entregas después de las 20:00 en Hora Local
  COUNTIF(hour_local >= 20) as deliveries_local_after_20,
  ROUND(COUNTIF(hour_local >= 20) / COUNT(*) * 100, 2) as pct_local_after_20
FROM delivery_events
GROUP BY country, timezone_offset
ORDER BY country;
"""

print("Analyzing UTC vs Local Late Deliveries (Timezone Illusion Investigation)...")
df_q5 = client.query(q5_timezone_query).to_dataframe()
df_q5

## QUESTION 6: Partner Consistency (PT-014 - SaoPauloShip)

Investigates the operational integrity of partner `PT-014` (SaoPauloShip), auditing route closure rates (stale `IN_PROGRESS` routes) and primary key duplication.

In [ ]:
q6_route_status_query = """
SELECT 
  partner_id,
  route_status,
  COUNT(*) as total_routes,
  SUM(estimated_stops) as total_estimated_stops,
  SUM(actual_stops) as total_actual_stops,
  ROUND(AVG(route_distance_km), 2) as avg_distance_km
FROM `meli-last-mile-sql-assessment.LAstmile.routes_new`
WHERE partner_id = 'PT-014'
GROUP BY partner_id, route_status;
"""

print("Querying route status breakdown for PT-014...")
df_q6_status = client.query(q6_route_status_query).to_dataframe()
df_q6_status

In [ ]:
q6_duplication_query = """
SELECT 
  COUNT(s.shipment_id) as total_raw_shipments,
  COUNT(DISTINCT s.shipment_id) as unique_shipments,
  COUNT(s.shipment_id) - COUNT(DISTINCT s.shipment_id) as duplicate_count,
  ROUND((COUNT(s.shipment_id) - COUNT(DISTINCT s.shipment_id)) / COUNT(s.shipment_id) * 100, 2) as duplicate_pct
FROM `meli-last-mile-sql-assessment.LAstmile.shipments_new` s
JOIN `meli-last-mile-sql-assessment.LAstmile.routes_new` r 
  ON s.route_id = r.route_id
WHERE r.partner_id = 'PT-014';
"""

print("Querying shipment duplication metrics for PT-014...")
df_q6_duplicates = client.query(q6_duplication_query).to_dataframe()
df_q6_duplicates